# 22. The stack, with the target-encoded neural model

**One variable against ledger row 32** (`stack_logit_23_oof`, CV 0.967764): the same
logistic combiner, the same `C`, the same fold-wise protocol, the same folds. The
member set goes from twenty-three to twenty-four, adding `neural_te` from row 33.

Row 32 is refit here from the same vectors rather than quoted, so the comparison is
paired per fold inside one run.

## Why this one is a different shape from the last two stack rows

Rows 27 and 32 added members that were **strong and redundant**: CatBoost, then four
more CatBoost seeds spanning 1.32e-05 of each other. The combiner priced them by
moving weight around, and the second of those was worth +0.000014.

`neural_te` is the opposite shape. It is **0.0015 weaker** than the best GBDT and
correlates at 0.9757 with CatBoost, near the bottom of the 0.9741 to 0.9981 band
LightGBM produces against **itself**. Strong enough to matter and still wrong in a
different direction is exactly the combination the 2026-08-12 correction says a
fitted combiner can use and an equal-weight one cannot.

## The old neural model stays in

`neural`, the raw-feature version from row 16, is **kept rather than replaced**. It is
a different model with a different error structure, it earns +0.1181 in row 32, and
removing it at the same time would make this two variables.

## What the coefficients would tell us

If `neural_te` takes a large weight **while `neural` keeps its own**, the two are wrong
in different directions from each other as well as from the trees. If `neural_te` takes
its weight straight out of `neural`, they are one mechanism at two strengths and the
weak one had been standing in for the strong one all along. The cell that prints them
reports the change in `neural`'s coefficient explicitly rather than leaving it to be
read off a table.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def find_repo():
    for b in [Path.cwd(), *Path.cwd().parents]:
        if (b / "data" / "raw" / "train.csv").exists():
            return b
    raise FileNotFoundError("data/raw/train.csv not found")


REPO = find_repo()
O, S = REPO / "artifacts" / "oof", REPO / "submissions"

train = pd.read_csv(REPO / "data" / "raw" / "train.csv")
test = pd.read_csv(REPO / "data" / "raw" / "test.csv")
y = train["addicted_label"].to_numpy()

# The same split every vector on disk was produced under. Rebuilt rather than loaded,
# and then checked, because a silently different fold vector is the one error here
# that produces a clean-looking wrong answer.
folds = np.full(len(train), -1, dtype=np.int64)
for i, (_, va) in enumerate(StratifiedKFold(5, shuffle=True,
                                            random_state=42).split(train, y)):
    folds[va] = i
assert (folds >= 0).all() and np.bincount(folds).sum() == len(train)
print(f"train {len(train):,}  test {len(test):,}  folds {np.bincount(folds)}")


train 691,369  test 296,302  folds [138274 138274 138274 138274 138273]


In [2]:
# Row 32's twenty-three in row 32's order, then the target-encoded neural model.
MEM = [
    ("te42", O / "te_bag42_oof.npy", O / "te_bag42_test.npy"),
    ("te2024", O / "te_seed2024_oof.npy", O / "te_seed2024_test.npy"),
    ("te7", O / "te_seed7_oof.npy", O / "te_seed7_test.npy"),
    ("te2025", O / "te_seed2025_oof.npy", O / "te_seed2025_test.npy"),
    ("te13", O / "te_seed13_oof.npy", O / "te_seed13_test.npy"),
    ("anchor", O / "lgbm_default_anchor_seed42.npy",
     S / "lgbm_default_anchor_seed42.csv"),
    ("trees300", O / "lgbm_trees300_seed42.npy", S / "lgbm_trees300_seed42.csv"),
    ("trees1000", O / "lgbm_trees1000_seed42.npy", S / "lgbm_trees1000_seed42.csv"),
    ("trees2000", O / "lgbm_trees2000_seed42.npy", S / "lgbm_trees2000_seed42.csv"),
    ("lr010", O / "lgbm_lr01_n1000_seed42.npy", S / "lgbm_lr01_n1000_seed42.csv"),
    ("lr005", O / "lgbm_lr005_n2000_seed42.npy", S / "lgbm_lr005_n2000_seed42.csv"),
    ("lr003", O / "lgbm_lr003_n3333_seed42.npy", S / "lgbm_lr003_n3333_seed42.csv"),
    ("bag42", O / "lgbm_bag08_lr005_n2000_seed42.npy",
     S / "lgbm_bag08_lr005_n2000_seed42.csv"),
    ("bag2024", O / "lgbm_bag08_lr005_n2000_seed2024.npy",
     S / "lgbm_bag08_lr005_n2000_seed2024.csv"),
    ("bag7", O / "lgbm_bag08_lr005_n2000_seed7.npy",
     S / "lgbm_bag08_lr005_n2000_seed7.csv"),
    ("bag2025", O / "lgbm_bag08_lr005_n2000_seed2025.npy",
     S / "lgbm_bag08_lr005_n2000_seed2025.csv"),
    ("bag13", O / "lgbm_bag08_lr005_n2000_seed13.npy",
     S / "lgbm_bag08_lr005_n2000_seed13.csv"),
    ("neural", O / "neural_oof.npy", O / "neural_test.npy"),
    ("cat42", O / "catboost_te_oof.npy", O / "catboost_te_test.npy"),
    ("cat2024", O / "catboost_te_seed2024_oof.npy",
     O / "catboost_te_seed2024_test.npy"),
    ("cat7", O / "catboost_te_seed7_oof.npy", O / "catboost_te_seed7_test.npy"),
    ("cat2025", O / "catboost_te_seed2025_oof.npy",
     O / "catboost_te_seed2025_test.npy"),
    ("cat13", O / "catboost_te_seed13_oof.npy", O / "catboost_te_seed13_test.npy"),
    ("neural_te", O / "neural_te_oof.npy", O / "neural_te_test.npy"),
]
NEW = ["neural_te"]
CATS = ["cat42", "cat2024", "cat7", "cat2025", "cat13"]


def logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-9, 1 - 1e-9)
    return np.clip(np.log(p / (1 - p)), -30, 30)


def load_test(path):
    if path.suffix == ".npy":
        return np.load(path)
    df = pd.read_csv(path)
    # A csv written in a different row order would blend perfectly cleanly and be
    # undetectable in the score. Checked rather than assumed.
    assert (df["id"].to_numpy() == test["id"].to_numpy()).all(), f"id order {path.name}"
    return df["addicted_label"].to_numpy()


names = [m[0] for m in MEM]
Poof = {n: np.load(p) for n, p, _ in MEM}
Ptest = {n: load_test(t) for n, _, t in MEM}

for n in names:
    assert Poof[n].shape == (len(train),), n
    assert Ptest[n].shape == (len(test),), n
    # A partially failed run leaves a constant fold, which blends silently.
    assert min(np.ptp(Poof[n][folds == f]) for f in range(5)) > 0, f"dead fold in {n}"

Loof = np.column_stack([logit(Poof[n]) for n in names])
Ltest = np.column_stack([logit(Ptest[n]) for n in names])
KEEP23 = [i for i, n in enumerate(names) if n not in NEW]
print(f"{len(names)} members, oof {Loof.shape}, test {Ltest.shape}")
print("member CV:")
for n in names:
    cv = np.mean([roc_auc_score(y[folds == f], Poof[n][folds == f]) for f in range(5)])
    print(f"  {n:10} {cv:.6f}" + ("   <- new" if n in NEW else ""))


24 members, oof (691369, 24), test (296302, 24)
member CV:


  te42       0.966782


  te2024     0.966771


  te7        0.966729


  te2025     0.966743


...

  cat2024    0.966928


  cat7       0.966920


  cat2025    0.966916


  cat13      0.966922


  neural_te  0.965373   <- new


In [3]:
# The fold loop, run twice over the identical folds: once on row 32's twenty-three
# and once with the target-encoded neural model added.
def run(cols):
    oof = np.zeros(len(train))
    tst = np.zeros((5, len(test)))
    cf = np.zeros((5, len(cols)))
    for f in range(5):
        tr, va = folds != f, folds == f
        clf = LogisticRegression(C=1.0, max_iter=2000).fit(Loof[np.ix_(tr, cols)],
                                                           y[tr])
        oof[va] = clf.decision_function(Loof[np.ix_(va, cols)])
        tst[f] = clf.decision_function(Ltest[:, cols])
        cf[f] = clf.coef_[0]
    per = np.array([roc_auc_score(y[folds == f], oof[folds == f]) for f in range(5)])
    return per, tst, cf


per23, test23, coef23 = run(KEEP23)
per24, test24, coef24 = run(list(range(len(names))))

print(f"{'':22} {'fold 0':>9} {'fold 1':>9} {'fold 2':>9} {'fold 3':>9} {'fold 4':>9}")
for lbl, p in (("23 members, row 32", per23), ("24 members, this run", per24)):
    print(f"{lbl:22} " + " ".join(f"{v:9.6f}" for v in p))
print()
print(f"23-member CV {per23.mean():.6f} +/- {per23.std():.6f}"
      f"   (row 32 recorded 0.967764 +/- 0.000434, "
      f"diff {per23.mean() - 0.967764:+.2e})")
print(f"24-member CV {per24.mean():.6f} +/- {per24.std():.6f}")


                          fold 0    fold 1    fold 2    fold 3    fold 4
23 members, row 32      0.967128  0.967916  0.968067  0.968305  0.967405
24 members, this run    0.967159  0.967964  0.968124  0.968325  0.967462

23-member CV 0.967764 +/- 0.000434   (row 32 recorded 0.967764 +/- 0.000434, diff +1.30e-08)
24-member CV 0.967807 +/- 0.000432


In [4]:
# Paired, on identical folds. The right test when two models share folds is the
# spread of the per-fold DIFFERENCES and how many folds it wins, not the fold spread,
# which is common to both and cancels.
def paired(a, b, lbl):
    d = a - b
    t = d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))
    print(f"{lbl:38} {d.mean():+.6f}  sd {d.std(ddof=1):.6f}  "
          f"{(d > 0).sum()}/5  t(4)={t:.2f}")
    print("     per fold: " + "  ".join(f"{v:+.6f}" for v in d))
    return d


base17 = np.array([roc_auc_score(y[folds == f], Poof["te42"][folds == f])
                   for f in range(5)])
d_new = paired(per24, per23, "24 vs 23 members (row 32)")
paired(per24, base17, "24-member vs te42 alone (row 17)")
print()
print("For scale, from this repo's own stack history: adding CatBoost to the")
print("18-member stack was worth +0.000100 (row 27), and adding four more CatBoost")
print("seeds on top of that was worth +0.000014 (row 32).")


24 vs 23 members (row 32)              +0.000043  sd 0.000016  5/5  t(4)=5.84
     per fold: +0.000031  +0.000048  +0.000056  +0.000020  +0.000057
24-member vs te42 alone (row 17)       +0.001024  sd 0.000052  5/5  t(4)=43.75
     per fold: +0.001049  +0.000988  +0.000951  +0.001061  +0.001072

For scale, from this repo's own stack history: adding CatBoost to the
18-member stack was worth +0.000100 (row 27), and adding four more CatBoost
seeds on top of that was worth +0.000014 (row 32).


In [5]:
# Coefficients. The question the header set up: does neural_te take its weight out of
# the old raw-feature neural model, or earn its own on top?
ROW32 = {"neural": 0.1181, "lr003": 0.1161, "lr005": 0.0998, "cat2024": 0.0939,
         "te13": 0.0919, "te2024": 0.0915, "te42": 0.0893, "te2025": 0.0891,
         "bag42": 0.0875, "bag7": 0.0817, "cat13": 0.0816, "cat2025": 0.0789,
         "bag13": 0.0784, "cat42": 0.0753, "cat7": 0.0720, "te7": 0.0711,
         "bag2025": 0.0590, "bag2024": 0.0143, "trees1000": 0.0074,
         "lr010": 0.0006, "trees2000": -0.0081, "trees300": -0.1505,
         "anchor": -0.3627}
print(f"{'member':10} {'mean':>9} {'sd across folds':>17}   {'row 32':>8}")
for i in np.argsort(-coef24.mean(axis=0)):
    was = ROW32.get(names[i])
    tag = "     new" if names[i] in NEW else (f"{was:>+8.4f}" if was is not None else "")
    print(f"{names[i]:10} {coef24[:, i].mean():>+9.4f} "
          f"{coef24[:, i].std():>17.4f}   {tag}")

i_old, i_new = names.index("neural"), names.index("neural_te")
old_now = coef24[:, i_old].mean()
print(f"\nold raw-feature neural : {old_now:+.4f}   was {ROW32['neural']:+.4f} in "
      f"row 32, change {old_now - ROW32['neural']:+.4f}")
print(f"new encoded neural     : {coef24[:, i_new].mean():+.4f}")
print("If the old one holds its weight, the two neural models are wrong in different")
print("directions from each other as well as from the trees. If it collapses, they")
print("are one mechanism at two strengths.")

cat_sum = sum(coef24[:, names.index(n)].mean() for n in CATS)
print(f"\nsum of the 5 CatBoost coefficients : {cat_sum:+.4f}   (row 32: +0.4016)")
print(f"largest fold-to-fold sd: {coef24.std(axis=0).max():.4f}")


member          mean   sd across folds     row 32
neural_te    +0.1305            0.0063        new
lr003        +0.1166            0.0171    +0.1161
lr005        +0.0954            0.0080    +0.0998
neural       +0.0905            0.0021    +0.1181
bag42        +0.0871            0.0107    +0.0875
te13         +0.0842            0.0074    +0.0919
bag7         +0.0826            0.0192    +0.0817
te2024       +0.0825            0.0140    +0.0915
te2025       +0.0819            0.0135    +0.0891
cat2024      +0.0799            0.0108    +0.0939
bag13        +0.0796            0.0088    +0.0784
te42         +0.0767            0.0077    +0.0893
cat13        +0.0657            0.0085    +0.0816
cat2025      +0.0650            0.0045    +0.0789
te7          +0.0612            0.0157    +0.0711
cat42        +0.0583            0.0032    +0.0753
bag2025      +0.0573            0.0064    +0.0590
cat7         +0.0524            0.0083    +0.0720
bag2024      +0.0167            0.0359    +0.0143


In [6]:
# Submission: the average of the five fold combiners, matching what every other
# submission in this repo does with its five fold models.
pred = test24.mean(axis=0)
prob = 1 / (1 + np.exp(-pred))

# Row 25 was held back because its ordering matched the submitted row 24 at Spearman
# 0.9999995. The same question decides whether this one is worth a slot.
prev = pd.read_csv(S / "stack_oof_23.csv")
assert (prev["id"].to_numpy() == test["id"].to_numpy()).all()
t32 = pd.Series(pred).corr(pd.Series(prev["addicted_label"].to_numpy()),
                           method="spearman")
t32b = pd.Series(pred).corr(pd.Series(test23.mean(axis=0)), method="spearman")
print(f"spearman vs the submitted row 32 : {t32:.7f}")
print(f"spearman vs row 32 refit here    : {t32b:.7f}")
print("Row 25 sat at 0.9999995 against row 24 and was held back on that basis.")

out = S / "stack_oof_24.csv"
sub = pd.DataFrame({"id": test["id"], "addicted_label": prob})
assert len(sub) == len(test) and sub["addicted_label"].between(0, 1).all()
sub.to_csv(out, index=False)
print(f"\nwrote {out.name}, {len(sub):,} rows, "
      f"range [{prob.min():.4f}, {prob.max():.4f}]")
print(f"ledger: CV {per24.mean():.6f} +/- {per24.std():.6f}, "
      f"vs row 32 {d_new.mean():+.6f} ({(d_new > 0).sum()}/5, "
      f"sd {d_new.std(ddof=1):.6f})")


spearman vs the submitted row 32 : 0.9998335
spearman vs row 32 refit here    : 0.9998335
Row 25 sat at 0.9999995 against row 24 and was held back on that basis.



wrote stack_oof_24.csv, 296,302 rows, range [0.0000, 1.0000]
ledger: CV 0.967807 +/- 0.000432, vs row 32 +0.000043 (5/5, sd 0.000016)
